In [6]:
# %pip install langchain-chroma
# %pip install pyprojroot
# %pip install langchain_huggingface
# %pip install sentence-transformers
# %pip install langchain
# %pip install langchain-classic

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [1]:
from pyprojroot import here
from langchain_community.utilities import SQLDatabase
from langchain_classic.chains import create_sql_query_chain
from langchain_community.tools.sql_database.tool import QuerySQLDataBaseTool
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from operator import itemgetter
from langchain_groq import ChatGroq
from langchain_core.tools import tool
import os
from dotenv import load_dotenv
load_dotenv()

/var/folders/s4/cxp4_1c1447576f31nclvwbh0000gp/T/ipykernel_20527/1672739693.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


True

**Set the environment variables and load the LLM**

In [2]:
os.environ['GROQ_API_KEY'] = os.getenv("GROQ_API_KEY")

llm = ChatGroq(model="qwen/qwen3.8-27b")

**Load and test the sqlite db**

In [3]:
sqldb_directory = here("data/travel.sqlite")
db = SQLDatabase.from_uri(
    f"sqlite:///{sqldb_directory}",
    sample_rows_in_table_info=0
    )

print(db.dialect)
print(db.get_usable_table_names())
db.run("SELECT * FROM aircrafts_data LIMIT 10;")


sqlite
['aircrafts_data', 'airports_data', 'boarding_passes', 'bookings', 'flights', 'seats', 'ticket_flights', 'tickets']


'[(\'773\', \'{"en": "Boeing 777-300", "ru": "Боинг 777-300"}\', 11100), (\'763\', \'{"en": "Boeing 767-300", "ru": "Боинг 767-300"}\', 7900), (\'SU9\', \'{"en": "Sukhoi Superjet-100", "ru": "Сухой Суперджет-100"}\', 3000), (\'320\', \'{"en": "Airbus A320-200", "ru": "Аэробус A320-200"}\', 5700), (\'321\', \'{"en": "Airbus A321-200", "ru": "Аэробус A321-200"}\', 5600), (\'319\', \'{"en": "Airbus A319-100", "ru": "Аэробус A319-100"}\', 6700), (\'733\', \'{"en": "Boeing 737-300", "ru": "Боинг 737-300"}\', 4200), (\'CN1\', \'{"en": "Cessna 208 Caravan", "ru": "Сессна 208 Караван"}\', 1200), (\'CR2\', \'{"en": "Bombardier CRJ-200", "ru": "Бомбардье CRJ-200"}\', 2700)]'

**Create the SQL agent chain and run a test query**

In [4]:
import re

def clean_sql(text: str) -> str:
    cleaned = re.sub(r"^.*?SQLQuery:\s*", "", text, flags=re.IGNORECASE)
    return cleaned.replace("```sql", "").replace("```", "").strip()

system_role = """Given the following user question, corresponding SQL query, 
    and SQL result, answer the user question.\n
    Question: {question}\n
    SQL Query: {query}\n
    SQL Result: {result}\n
    Answer:
    """

execute_query = QuerySQLDataBaseTool(db=db)
write_query = create_sql_query_chain(llm, db) | RunnableLambda(clean_sql)
answer_prompt = PromptTemplate.from_template(system_role)
answer = answer_prompt | llm | StrOutputParser()
chain = (
    RunnablePassthrough.assign(query=write_query).assign(
        result=itemgetter("query") | execute_query
    ) | answer
)

/var/folders/s4/cxp4_1c1447576f31nclvwbh0000gp/T/ipykernel_20527/3439367733.py:15: LangChainDeprecationWarning: The class `QuerySQLDataBaseTool` was deprecated in LangChain 0.3.12 and will be removed in 1.0. An updated version of the class exists in the `langchain-community package and should be used instead. To use it run `pip install -U `langchain-community` and import as `from `langchain_community.tools import QuerySQLDatabaseTool``.
  execute_query = QuerySQLDataBaseTool(db=db)


In [5]:
message = "How many tables do I have in the database? and what are their names?"
response = chain.invoke({"question": message})
response

'You have 8 tables in the database. Their names are:\n\n*   aircrafts_data\n*   airports_data\n*   boarding_passes\n*   bookings\n*   flights\n*   seats\n*   ticket_flights\n*   tickets'

**Travel SQL-agent Tool Design**

In [6]:
import re
class TravelSQLAgentTool:
    """
    A tool for interacting with a travel-related SQL database using an LLM (Language Model) to generate and execute SQL queries.

    This tool enables users to ask travel-related questions, which are transformed into SQL queries by a language model.
    The SQL queries are executed on the provided SQLite database, and the results are processed by the language model to
    generate a final answer for the user.

    Attributes:
        sql_agent_llm (ChatOpenAI): An instance of a ChatOpenAI language model used to generate and process SQL queries.
        system_role (str): A system prompt template that guides the language model in answering user questions based on SQL query results.
        db (SQLDatabase): An instance of the SQL database used to execute queries.
        chain (RunnablePassthrough): A chain of operations that creates SQL queries, executes them, and generates a response.

    Methods:
        __init__: Initializes the TravelSQLAgentTool by setting up the language model, SQL database, and query-answering pipeline.
    """


    def clean_sql(text: str) -> str:
        cleaned = re.sub(r"^.*?SQLQuery:\s*", "", text, flags=re.IGNORECASE)
        return cleaned.replace("```sql", "").replace("```", "").strip()
    
    def __init__(self, llm: str, sqldb_directory: str, llm_temerature: float) -> None:
        """
        Initializes the TravelSQLAgentTool with the necessary configurations.

        Args:
            llm (str): The name of the language model to be used for generating and interpreting SQL queries.
            sqldb_directory (str): The directory path where the SQLite database is stored.
            llm_temerature (float): The temperature setting for the language model, controlling response randomness.
        """
        self.sql_agent_llm = ChatGroq(
            model=llm, temperature=llm_temerature)
        self.system_role = """Given the following user question, corresponding SQL query, and SQL result, answer the user question.\n
            Question: {question}\n
            SQL Query: {query}\n
            SQL Result: {result}\n
            Answer:
            """
        self.db = SQLDatabase.from_uri(
            f"sqlite:///{sqldb_directory}")
        print(self.db.get_usable_table_names())

        execute_query = QuerySQLDataBaseTool(db=self.db)
        write_query = create_sql_query_chain(llm, db) | RunnableLambda(clean_sql)
        answer_prompt = PromptTemplate.from_template(self.system_role)

        answer = answer_prompt | self.sql_agent_llm | StrOutputParser()
        self.chain = (
            RunnablePassthrough.assign(query=write_query).assign(
                result=itemgetter("query") | execute_query)| answer
        )

In [7]:
import sys
from pyprojroot import here
root_dir = str(here())
if root_dir not in sys.path:
    sys.path.append(root_dir)
    
from src.agent_graph.load_tools_config import LoadToolsConfig

TOOLS_CFG = LoadToolsConfig()

@tool
def query_travel_sqldb(query: str) -> str:
    """Query the Swiss Airline SQL Database and access all the company's information. Input should be a search query."""
    agent = TravelSQLAgentTool(
        llm=TOOLS_CFG.travel_sqlagent_llm,
        sqldb_directory=TOOLS_CFG.travel_sqldb_directory,
        llm_temerature=TOOLS_CFG.travel_sqlagent_llm_temperature
    )
    response = agent.chain.invoke({"question": query})
    return response